# import

In [ ]:
from utils.train_evaluate import recbole_train_each_eval_all, get_knn_method
from utils.plot_utils import get_full_models_results_matrix, plot_and_save_heatmap
from utils.file_utils import load_picklefile
from utils.transfer_learning_scores import * 
import pandas as pd

IS_TO_TRAIN_MODELS = True

# Experiment Goodreads - PT ND LS.t UD SF TO UM.100
pre train, no drift, leave one out train sampled, uniform neg sample training distribution, shuffle false, time ordering, uniform mode


In [ ]:
save_path, base_filename, specs_str = ('processed_datasets/natural_data/goodreads/two_intervals/',
                                        'more_2interQ_df', 
                                        'PT_ND')
BENCHMARK_FILENAMES = ['train', 'valid', 'test']
base_dataset_name = base_filename+'_'+specs_str

# variables

In [ ]:
images_filepath='images/goodreads_two_intervals/'

freq=12 # month
duration = 2*12//freq # 2 years split in xM buckets
n_parts = duration*2+1
d_keys = ['_pt'+str(i) for i in range(1, n_parts)]

# MODEL_VERSIONS = ['_pt1', '_pt2']
MODEL_VERSIONS = d_keys[:duration]

PART_SHIFT_INCL = False # include pt 8 (interval that has the before and after the shift)

Ks = [1, 10, 20]
VM_K = Ks[2] # valid metric k, also used in heatmap matrix
VALID_METRIC = 'Recall@'+str(VM_K)
SEED = 2020
USE_GPU = True # CHANGE FILENAME VERSION TOO!!!
SHOW_PROGRESS = False # useful when debug
SAVE_DATASET = False 

# dataset_save_path (str): The path of saved dataset. The tool will attempt to load the dataset from this path. If it equals to None, the tool will try to load the dataset from {checkpoint_dir}/{dataset}-{dataset_class_name}.pth.

# these are the default values
# TRAIN_NEG_SAMPLE_ARGS = {'distribution': 'uniform', 
#                          'sample_num': 1, 
#                          'alpha': 1.0, 
#                          'dynamic': False, 
#                          'candidate_num': 0}



SHUFFLE = False  # shuffle (bool): Whether or not to shuffle the training data before each epoch. Defaults to True.
EVAL_ARGS = {'split': {'LS': 'test_only'}, # leave-one-out sample type ['valid_and_test', 'valid_only', 'test_only']
                    'group_by': 'user',
                    'order': 'TO', # order (str): decides how we sort the data in .inter. random ordering or time ordering
                    'mode': 'uni100'}

METRICS = ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision'] 
# both 'GiniIndex' and 'TailPercentage' need "rec.items" and "data.count_items"
# Recall, MRR, NDCG, Hit, MAP, Precision, GAUC, ItemCoverage, AveragePopularity, GiniIndex, ShannonEntropy, TailPercentage

FILENAME_VERSION = '_JRC_GPU_LS.t_UD_SF_TO_UM.100'

# BPR

In [ ]:
%%time
if IS_TO_TRAIN_MODELS:
    model_name = 'BPR'
    knn_method = None
    recbole_train_each_eval_all(model_name=model_name,
                                knn_method=knn_method,
                                model_versions_to_train=MODEL_VERSIONS,
                                model_versions_to_evaluate=MODEL_VERSIONS,
                                save_path=save_path,
                                base_filename=base_filename,
                                specs_str=specs_str,
                                filename_version=FILENAME_VERSION,

                                use_gpu=USE_GPU,
                                seed=SEED,
                                show_progress=SHOW_PROGRESS,
                                save_dataset=SAVE_DATASET,
                                shuffle=SHUFFLE,
                                benchmark_filename=BENCHMARK_FILENAMES,
                                eval_args=EVAL_ARGS,
                                metrics=METRICS,
                                ks=Ks,
                                valid_metric=VALID_METRIC,
                            
                                part_shift_incl=PART_SHIFT_INCL)


## recall heatmap

In [ ]:
model_name = 'BPR'

results_matrix = get_full_models_results_matrix(model_name=model_name,
                                    models_versions=MODEL_VERSIONS,
                                    base_dataset_name=base_dataset_name,
                                    save_path=save_path,
                                    metric='recall@'+str(VM_K),
                                    filename_version=FILENAME_VERSION)

plot_and_save_heatmap(results_matrix,
               round_point=4,
               title=model_name+' - Goodreads - No Drift - '+VALID_METRIC, 
               filepath=images_filepath,
               filename=base_dataset_name+FILENAME_VERSION+'_'+model_name+'_'+VALID_METRIC+'.jpg')

## transfer scores

In [ ]:
# model_name = 'BPR'

# results_matrix = get_full_models_results_matrix(model_name=model_name,
#                                     models_versions=MODEL_VERSIONS,
#                                     base_dataset_name=base_dataset_name,
#                                     save_path=save_path,
#                                     metric='recall@'+str(VM_K),
#                                     filename_version=FILENAME_VERSION)

# bwt, _ = compute_BWT_rodrigues(results_matrix)
# sbwt, _ = compute_symmetric_BWT_rodrigues(results_matrix)

# bwt3md, _ = compute_BWT_rodrigues_three_main_diagonals(results_matrix)
# sbwt3md, _ = compute_symmetric_BWT_rodrigues_three_main_diagonals(results_matrix)

# print('Considering the complete triangle\nBWT=', bwt)
# print('Symmetric BWT (corrected FWT)=', sbwt)
# print('\n\nConsidering the 3 main diagonals\nBWT=', bwt3md)
# print('Symmetric BWT (corrected FWT)=', sbwt3md)

# NeuMF

In [ ]:
%%time 
if IS_TO_TRAIN_MODELS:
    model_name = 'NeuMF'
    knn_method = None
    recbole_train_each_eval_all(model_name=model_name,
                                knn_method=knn_method,
                                model_versions_to_train=MODEL_VERSIONS,
                                model_versions_to_evaluate=MODEL_VERSIONS,
                                save_path=save_path,
                                base_filename=base_filename,
                                specs_str=specs_str,
                                filename_version=FILENAME_VERSION,

                                use_gpu=USE_GPU,
                                seed=SEED,
                                show_progress=SHOW_PROGRESS,
                                save_dataset=SAVE_DATASET,
                                shuffle=SHUFFLE,
                                benchmark_filename=BENCHMARK_FILENAMES,
                                eval_args=EVAL_ARGS,
                                metrics=METRICS,
                                ks=Ks,
                                valid_metric=VALID_METRIC,
                            
                                part_shift_incl=PART_SHIFT_INCL)

## recall heatmap

In [ ]:
model_name = 'NeuMF'

results_matrix = get_full_models_results_matrix(model_name=model_name,
                                    models_versions=MODEL_VERSIONS,
                                    base_dataset_name=base_dataset_name,
                                    save_path=save_path,
                                    metric='recall@'+str(VM_K),
                                    filename_version=FILENAME_VERSION)

plot_and_save_heatmap(results_matrix,
               round_point=4,
               title=model_name+' - Goodreads - No Drift - '+VALID_METRIC, 
               filepath=images_filepath,
               filename=base_dataset_name+FILENAME_VERSION+'_'+model_name+'_'+VALID_METRIC+'.jpg')

## transfer scores

In [ ]:
# model_name = 'NeuMF'

# results_matrix = get_full_models_results_matrix(model_name=model_name,
#                                     models_versions=MODEL_VERSIONS,
#                                     base_dataset_name=base_dataset_name,
#                                     save_path=save_path,
#                                     metric='recall@'+str(VM_K),
#                                     filename_version=FILENAME_VERSION)

# bwt, _ = compute_BWT_rodrigues(results_matrix)
# sbwt, _ = compute_symmetric_BWT_rodrigues(results_matrix)

# bwt3md, _ = compute_BWT_rodrigues_three_main_diagonals(results_matrix)
# sbwt3md, _ = compute_symmetric_BWT_rodrigues_three_main_diagonals(results_matrix)

# print('Considering the complete triangle\nBWT=', bwt)
# print('Symmetric BWT (corrected FWT)=', sbwt)
# print('\n\nConsidering the 3 main diagonals\nBWT=', bwt3md)
# print('Symmetric BWT (corrected FWT)=', sbwt3md)

# Pop

In [ ]:
%%time 
if IS_TO_TRAIN_MODELS:
    model_name = 'Pop'
    knn_method = None
    recbole_train_each_eval_all(model_name=model_name,
                                knn_method=knn_method,
                                model_versions_to_train=MODEL_VERSIONS,
                                model_versions_to_evaluate=MODEL_VERSIONS,
                                save_path=save_path,
                                base_filename=base_filename,
                                specs_str=specs_str,
                                filename_version=FILENAME_VERSION,

                                use_gpu=USE_GPU,
                                seed=SEED,
                                show_progress=SHOW_PROGRESS,
                                save_dataset=SAVE_DATASET,
                                shuffle=SHUFFLE,
                                benchmark_filename=BENCHMARK_FILENAMES,
                                eval_args=EVAL_ARGS,
                                metrics=METRICS,
                                ks=Ks,
                                valid_metric=VALID_METRIC,
                            
                                part_shift_incl=PART_SHIFT_INCL)

## recall heatmap

In [ ]:
model_name = 'Pop'

results_matrix = get_full_models_results_matrix(model_name=model_name,
                                    models_versions=MODEL_VERSIONS,
                                    base_dataset_name=base_dataset_name,
                                    save_path=save_path,
                                    metric='recall@'+str(VM_K),
                                    filename_version=FILENAME_VERSION)

plot_and_save_heatmap(results_matrix,
               round_point=4,
               title=model_name+' - Goodreads - No Drift - '+VALID_METRIC, 
               filepath=images_filepath,
               filename=base_dataset_name+FILENAME_VERSION+'_'+model_name+'_'+VALID_METRIC+'.jpg')

## transfer scores

In [ ]:
# model_name = 'Pop'

# results_matrix = get_full_models_results_matrix(model_name=model_name,
#                                     models_versions=MODEL_VERSIONS,
#                                     base_dataset_name=base_dataset_name,
#                                     save_path=save_path,
#                                     metric='recall@'+str(VM_K),
#                                     filename_version=FILENAME_VERSION)

# bwt, _ = compute_BWT_rodrigues(results_matrix)
# sbwt, _ = compute_symmetric_BWT_rodrigues(results_matrix)

# bwt3md, _ = compute_BWT_rodrigues_three_main_diagonals(results_matrix)
# sbwt3md, _ = compute_symmetric_BWT_rodrigues_three_main_diagonals(results_matrix)

# print('Considering the complete triangle\nBWT=', bwt)
# print('Symmetric BWT (corrected FWT)=', sbwt)
# print('\n\nConsidering the 3 main diagonals\nBWT=', bwt3md)
# print('Symmetric BWT (corrected FWT)=', sbwt3md)

# ItemKNN

In [ ]:
# more_2interQ_df = pd.read_csv(save_path+base_filename+'.csv')
# print('more_2interQ_df')
# get_knn_method(more_2interQ_df);



# pt4_train = pd.read_csv(save_path+base_filename+'_'+specs_str+MODEL_VERSIONS[-1]+'/'+base_filename+'_'+specs_str+MODEL_VERSIONS[-1]+'.train.csv')
# print('\n\npt4_train with pre train')
# get_knn_method(pt4_train);

In [ ]:
# knn_method = get_knn_method(pt4_train)

In [ ]:
%%time

if IS_TO_TRAIN_MODELS:
    model_name = 'ItemKNN'
    knn_method = 'user'

    recbole_train_each_eval_all(model_name=model_name,
                                knn_method=knn_method,
                                model_versions_to_train=MODEL_VERSIONS,
                                model_versions_to_evaluate=MODEL_VERSIONS,
                                save_path=save_path,
                                base_filename=base_filename,
                                specs_str=specs_str,
                                filename_version=FILENAME_VERSION,

                                use_gpu=USE_GPU,
                                seed=SEED,
                                show_progress=SHOW_PROGRESS,
                                save_dataset=SAVE_DATASET,
                                shuffle=SHUFFLE,
                                benchmark_filename=BENCHMARK_FILENAMES,
                                eval_args=EVAL_ARGS,
                                metrics=METRICS,
                                ks=Ks,
                                valid_metric=VALID_METRIC,
                            
                                part_shift_incl=PART_SHIFT_INCL)

## recall heatmap

In [ ]:
model_name = 'ItemKNN'
knn_method = 'user'

results_matrix = get_full_models_results_matrix(model_name=model_name,
                                    models_versions=MODEL_VERSIONS,
                                    base_dataset_name=base_dataset_name,
                                    save_path=save_path,
                                    metric='recall@'+str(VM_K),
                                    filename_version=FILENAME_VERSION)

plot_and_save_heatmap(results_matrix,
               round_point=4,
               title='UserKNN'+' - Goodreads - No Drift - '+VALID_METRIC, 
               filepath=images_filepath,
               filename=base_dataset_name+FILENAME_VERSION+'_'+'UserKNN'+'_'+VALID_METRIC+'.jpg')

## transfer scores

In [ ]:
# model_name = 'ItemKNN'

# results_matrix = get_full_models_results_matrix(model_name=model_name,
#                                     models_versions=MODEL_VERSIONS,
#                                     base_dataset_name=base_dataset_name,
#                                     save_path=save_path,
#                                     metric='recall@'+str(VM_K),
#                                     filename_version=FILENAME_VERSION)

# bwt, _ = compute_BWT_rodrigues(results_matrix)
# sbwt, _ = compute_symmetric_BWT_rodrigues(results_matrix)

# bwt3md, _ = compute_BWT_rodrigues_three_main_diagonals(results_matrix)
# sbwt3md, _ = compute_symmetric_BWT_rodrigues_three_main_diagonals(results_matrix)

# print('Considering the complete triangle\nBWT=', bwt)
# print('Symmetric BWT (corrected FWT)=', sbwt)
# print('\n\nConsidering the 3 main diagonals\nBWT=', bwt3md)
# print('Symmetric BWT (corrected FWT)=', sbwt3md)